- In here use nawaloka hospital website to crawl data : https://www.nawaloka.com 
- Save markdown files and consolidated JSONL corpus.


### 1. Import environment setup

In [1]:
import os
import sys
import json
import time
import nest_asyncio

from pathlib import Path
from urllib.parse import urlparse
from dotenv import load_dotenv

nest_asyncio.apply()

In [2]:
# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

load_dotenv(project_root / ".env")

# Check for API key (OpenRouter preferred, OpenAI as fallback)
openrouter_key = os.getenv("OPENROUTER_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

if not openrouter_key and not openai_key:
    raise EnvironmentError(
        "No API key found!\n"
        "Add OPENROUTER_API_KEY (recommended) or OPENAI_API_KEY to .env"
    )

provider = "OpenRouter" if openrouter_key else "OpenAI"
print("Environment loaded")
print(f"Provider: {provider}")
print(f"Project root: {project_root}")

Environment loaded
Provider: OpenRouter
Project root: d:\portfolio projects\hospital-system


### 2. Import configurations

In [12]:
from utils.config_utils import *
from application.ingest_doc import *

CRAWL_OUT_DIR = Path("data")
MARKDOWN_DIR = Path("data/nawaloka_markdown").parent

MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)


### 3. Crawl Configurations

In [4]:
BASE_URL = "https://www.nawaloka.com"

START_PATHS = [
    "/", "/our-centres", "/healthchecks", "/channeling",
    "/aboutus", "/contactus", "/blogs-and-news", "/emergency",
    "/international", "/our-centres/cardiology", "/our-centres/oncology",
    "/our-centres/neurology", "/our-centres/orthopedics",
    "/our-centres/pediatrics", "/healthchecks/executive"
]

START_URLS = [BASE_URL + path for path in START_PATHS]

EXCLUDE_PATTERNS = [
    "/login", "/terms", "/privacy", "/admin",
    "/images/", "/downloads/", "/media/"
]

MAX_DEPTH = 3
REQUEST_DELAY = 2.0
JSONL_PATH = CRAWL_OUT_DIR / "nawaloka_docs.jsonl"

print(f"Crawl config:")
print(f"   - Start URLs: {len(START_URLS)}")
print(f"   - Max depth: {MAX_DEPTH}")
print(f"   - Request delay: {REQUEST_DELAY}s")

Crawl config:
   - Start URLs: 15
   - Max depth: 3
   - Request delay: 2.0s


### 4. Execute crawling

In [5]:
start_time = time.time()

crawler = WebCrawler(
    base_url=BASE_URL,
    max_depth=MAX_DEPTH,
    exclude_patterns=EXCLUDE_PATTERNS
)

print(f"\nStarting crawl at {time.strftime('%H:%M:%S')}\n")
documents = crawler.crawl(START_URLS, request_delay=REQUEST_DELAY)

elapsed = time.time() - start_time
print(f"\nCrawl complete in {elapsed:.1f}s")
print(f"Documents collected: {len(documents)}")
print(f"URLs visited: {len(crawler.visited)}")

2026-11-05 23:48:38,357 - application.ingest_doc.web_crawler - INFO - Initialize the BFS queue
2026-11-05 23:48:38,361 - application.ingest_doc.web_crawler - INFO - Starting crawl with 15 seed URLs



Starting crawl at 23:48:38



2026-11-05 23:48:38,958 - application.ingest_doc.web_crawler - INFO - Launching browser......
2026-11-05 23:48:39,398 - application.ingest_doc.web_crawler - INFO - Browser launched successfully (channel=chrome)
2026-11-05 23:48:40,030 - application.ingest_doc.web_crawler - INFO - [0] Crawling: https://www.nawaloka.com/
2026-11-05 23:48:51,651 - application.ingest_doc.web_crawler - INFO - Starting extracting .........
2026-11-05 23:48:51,652 - application.ingest_doc.web_crawler - INFO - step 1 : Remove unessential tags 

2026-11-05 23:48:51,658 - application.ingest_doc.web_crawler - INFO - step 2 : Extract title from the page 

2026-11-05 23:48:51,659 - application.ingest_doc.web_crawler - INFO - step 3 : Extract all content inside the h tags 

2026-11-05 23:48:51,666 - application.ingest_doc.web_crawler - INFO - step 4 : Extract all content inside the h tags 

2026-11-05 23:48:51,671 - application.ingest_doc.web_crawler - INFO - step 5 : Convert html to markdown 

2026-11-05 23:48:51,8


Crawl complete in 673.0s
Documents collected: 55
URLs visited: 68


In [13]:
for i, doc in enumerate(documents):
    url_path = urlparse(doc['url']).path.strip('/').replace('/', '_')
    if not url_path:
        url_path = "homepage"
    filename = f"{i:03d}_{url_path}.md"
    
    md_file = MARKDOWN_DIR / filename
    
    with open(md_file, 'w', encoding='utf-8') as f:
        f.write(f"# {doc['title']}\n\n")
        f.write(f"**URL**: {doc['url']}\n\n")
        f.write(f"**Depth**: {doc['depth_level']}\n\n")
        f.write("---\n\n")
        f.write(doc['content'])

print(f"Saved {len(documents)} markdown files to {MARKDOWN_DIR}")

# Save JSONL
with open(JSONL_PATH, 'w', encoding='utf-8') as f:
    for doc in documents:
        f.write(json.dumps(doc, ensure_ascii=False) + '\n')

print(f"Saved JSONL corpus to {JSONL_PATH}")

Saved 55 markdown files to data
Saved JSONL corpus to data\nawaloka_docs.jsonl


In [15]:
# Cell 8: Quality Checks
import random

print("Quality Checks:\n")

# Check markdown files
md_files = list(MARKDOWN_DIR.glob("*.md"))
print(f"Markdown files: {len(md_files)}")

if len(md_files) >= 20:
    print(f"Good! Got {len(md_files)} pages")
elif len(md_files) >= 10:
    print(f"Only {len(md_files)} pages (site may be small)")
else:
    raise AssertionError(f"Too few pages: {len(md_files)}")

# Check JSONL
assert JSONL_PATH.exists(), f"JSONL not found"
print(f"\n2️JSONL file: {JSONL_PATH.stat().st_size:,} bytes")

# Sample inspection
with open(JSONL_PATH, 'r', encoding='utf-8') as f:
    all_docs = [json.loads(line) for line in f]

samples = random.sample(all_docs, min(3, len(all_docs)))
print(f"\n3️Random samples:\n")

for i, doc in enumerate(samples, 1):
    print(f"   Sample {i}:")
    print(f"   - URL: {doc['url']}")
    print(f"   - Title: {doc['title']}")
    print(f"   - Words: {len(doc['content'].split())}")
    print()

print("All quality checks passed!")

Quality Checks:

Markdown files: 55
Good! Got 55 pages

2️JSONL file: 2,134,315 bytes

3️Random samples:

   Sample 1:
   - URL: https://www.nawaloka.com/doctorprofile/D1111111
   - Title: Channel Dr. Hemant Digambar Waikar - Nawaloka Hospital PLC - Colombo 02
   - Words: 630

   Sample 2:
   - URL: https://www.nawaloka.com/lasik-advanced-eye-care-centre
   - Title: Eye Surgery in Sri Lanka | LASIK Eye Treatment | Nawaloka | Nawaloka Hospitals PLC
   - Words: 1301

   Sample 3:
   - URL: https://www.nawaloka.com/blogs-and-news/nawaloka-hospitals-colombo-mobile-dengue-testing-clinical-capacity
   - Title: Mobile Dengue Testing in Sri Lanka | Nawaloka Hospitals
   - Words: 915

All quality checks passed!
